# 04 — Approval policy simulation

**The core notebook.** A probability cannot be executed; it has to become *approve above this cut-off, decline below that one*. Everything before this was preparation.

In [ ]:
import sys; sys.path.insert(0, '..')
import warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from src.config import load_config

cfg = load_config()
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 60)

import pickle
from src.splits import assign_split
from src.data_load import months_of_credit_history
from src.models import simple_rule_score
from src.policy import compare_policies, grade_to_score, rough_economics
from src.metrics import summary_metrics, calibration_table

df = pd.read_parquet('../data/processed/modelling_sample.parquet')
df['credit_history_months'] = months_of_credit_history(df['earliest_cr_line'], df['issue_dt'])
df['split'] = assign_split(df, cfg)
va, te = df[df['split']=='valid'], df[df['split']=='test']
with open('../data/processed/fitted.pkl','rb') as f:
    fit = pickle.load(f)
sc, gbm, NUM = fit['scorecard'], fit['gbm'], fit['NUM']

def build(frame):
    s = {'tier1_rule': simple_rule_score(frame),
         'tier2_scorecard': sc.predict_proba(frame),
         'tier3_gbm': gbm.predict_proba(frame[NUM])[:,1],
         'platform_sub_grade': grade_to_score(frame['sub_grade']).to_numpy()}
    s['stacked'] = (pd.Series(s['platform_sub_grade']).rank(pct=True).to_numpy()
                    + pd.Series(s['tier2_scorecard']).rank(pct=True).to_numpy()) / 2
    return s

## 1. The trade-off curve

The cut-off is a **quantile of the score**, so approval rate is held exactly and bad rate is the free variable. That is the direction the business question runs: the lender decides how much to lend, then discovers what it costs.

In [ ]:
cmp = compare_policies(va['target'].to_numpy(), build(va), cfg['policy']['approval_rates'])
cmp.pivot(index='approval_rate', columns='policy', values='approved_bad_rate')

## 2. The column to lead with

Reporting defaults-avoided without goods-rejected is selling something. At 70% approval the policy stops about half the defaults and turns away a quarter of the people who would have repaid; only ~26% of declined applicants would actually have defaulted.

In [ ]:
cmp[cmp['policy']=='stacked'][['approval_rate','approved_bad_rate',
    'bads_avoided_share','goods_rejected_share','precision_of_decline']]

## 3. Rough economics — crude by design

No funding cost, no servicing cost, no prepayment, no collections model, flat assumed recovery. It ranks policies on shared assumptions; it does **not** estimate profit. At 70% approval on test the net proxy is 0.22% of capital deployed — essentially break-even before any real cost. Useful as scale discipline, not as a finding.

In [ ]:
st = build(te)
order = np.argsort(st['stacked'])
mask = np.zeros(len(te), bool); mask[order[:int(len(te)*0.7)]] = True
rough_economics(te.reset_index(drop=True), mask, cfg)

## 4. TEST — first and only evaluation

Everything above was tuned on validation. This cell runs once.

**Verified result:** stacked beats the platform grade by 30–37 bp at every approval rate from 50% to 80%, and the sign held from validation through to test (−38/−35/−36/−29/−17 bp on valid, −32/−33/−37/−30/−13 bp on test).

In [ ]:
ct = compare_policies(te['target'].to_numpy(), st, cfg['policy']['approval_rates'])
pt = ct.pivot(index='approval_rate', columns='policy', values='approved_bad_rate')
print(pt.to_string())
print('\nvs platform, basis points (negative = better):')
for c in ['tier2_scorecard','tier3_gbm','stacked']:
    print(f'  {c:<20}', ((pt[c]-pt['platform_sub_grade'])*10000).round(1).tolist())

## 5. Calibration

Ranking and calibration are different properties, and a threshold policy needs both. See 05 for why this one matters more than the PSI.

In [ ]:
calibration_table(te['target'].to_numpy(), st['tier2_scorecard'])